In [1]:
import pandas as pd
import re
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import json
from pathlib import Path
import numpy as np
from tqdm.auto import tqdm
import torch
import torch.nn.functional as F

/Users/mnatali/Projects/sentiment_analysis/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version 2.9.1 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
W0729 11:26:51.489000 77607 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
language_classifier = pipeline(
    "text-classification",
    model = "papluca/xlm-roberta-base-language-detection"
)

def lang_result(text):
    results = language_classifier(
        text,
        truncation=True
    )
    return results[0]["label"]

theme_classifier = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/deberta-v3-base-zeroshot-v2.0",
    multi_label = True
)

def theme_result(text, theme_labels):
    return theme_classifier(
        text,
        candidate_labels=theme_labels,
        hypothesis_template="This post discusses {}.",
        multi_label=True
    )

model_name = "yangheng/deberta-v3-base-absa-v1.1"

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
model.eval()


def aspect_sentiment(text, aspect, batch_size=16, max_length=512, stride=64):
    encoded = tokenizer(
        text,
        aspect,
        truncation=True,
        max_length=max_length,
        stride=stride,
        return_overflowing_tokens=True,
        padding=True,
        return_tensors="pt"
    )

    input_keys = ["input_ids", "attention_mask", "token_type_ids"]
    input_keys = [k for k in input_keys if k in encoded]

    all_probs = []

    with torch.inference_mode():
        n_chunks = encoded["input_ids"].shape[0]

        for start in range(0, n_chunks, batch_size):
            end = start + batch_size

            batch = {
                k: encoded[k][start:end].to(device)
                for k in input_keys
            }

            outputs = model(**batch)
            probs = F.softmax(outputs.logits, dim=-1)
            all_probs.append(probs)

    avg_probs = torch.cat(all_probs, dim=0).mean(dim=0).cpu()

    return {
        model.config.id2label[i]: float(avg_probs[i])
        for i in range(len(avg_probs))
    }

Device set to use mps:0
Device set to use mps:0


Using device: mps


/Users/mnatali/Projects/sentiment_analysis/.venv/lib/python3.13/site-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [3]:
def remove_links(text):
    url_pattern = re.compile(r'\[https?://\S+\]|\(https?://\S+\)|\[www\.\S+\]|\(www\.\S+\)')
    cleaned_text = url_pattern.sub('', text)
    return cleaned_text

In [4]:
BASE_DIR = Path.cwd()
file_path = (
    BASE_DIR
    / "brightdata_social_exports"
    / "x_datacenters_posts.json"
)
with file_path.open("r", encoding="utf-8") as f:
    x_posts = json.load(f)

In [5]:
english_post_ids = []
a = 0

for x_post in x_posts:
    unclean_text = x_post["description"]
    text = remove_links(unclean_text)
    language = lang_result(text)
    pid = x_post["id"]
    if language == 'en':
        english_post_ids.append(pid)
    a += 1
    print("Posts scanned:", a, end="\r")

KeyboardInterrupt: 

In [6]:
print(len(english_post_ids))

11


In [7]:
all_post_ids = []

env_post_ids = []
env_post_sentiments = []
env_post_sentiment_degrees = []

infr_post_ids = []
infr_post_sentiments = []
infr_post_sentiment_degrees = []

housing_post_ids = []
housing_post_sentiments = []
housing_post_sentiment_degrees = []

econ_post_ids = []
econ_post_sentiments = []
econ_post_sentiment_degrees = []

life_qual_post_ids = []
life_qual_post_sentiments = []
life_qual_post_sentiment_degrees = []

aesth_post_ids = []
aesth_post_sentiments = []
aesth_post_sentiment_degrees = []

gov_post_ids = []
gov_post_sentiments = []
gov_post_sentiment_degrees = []

tech_post_ids = []
tech_post_sentiments = []
tech_post_sentiment_degrees = []

not_useful_post_ids = []

themes = ["visual impact of datacenters", "infrastructure and house utilities", "housing costs and property values", "economy and jobs", "quality of life, noise, and light pollution", "environmental impact", "government decisions and policies", "technology performance and growth"]

matched_posts = 0
more_than_one_theme_posts = 0

a = 0

for x_post in x_posts:

    post_id = x_post["id"]
    if post_id not in english_post_ids:
        continue
    all_post_ids.append(post_id)
    unclean_text = x_post["description"]
    text = remove_links(unclean_text)


    final_labels = []

    theme_scores = theme_result(
        text,
        themes
    )
    
    for i in range(len(theme_scores['labels'])):
        if theme_scores['scores'][i] > 0.5:
            final_labels.append(theme_scores['labels'][i])
    
    if len(final_labels) > 0:
        matched_posts += 1
    else:
        not_useful_post_ids.append(post_id)
    
    if len(final_labels) > 1:
        more_than_one_theme_posts += 1
    

    for label in final_labels:
        total_sentiment = aspect_sentiment(text, label)
        post_sentiment = max(total_sentiment, key=total_sentiment.get)
        post_degree = max(total_sentiment.values())

        if label == "visual impact of datacenters":
            aesth_post_ids.append(post_id)
            aesth_post_sentiments.append(post_sentiment)
            aesth_post_sentiment_degrees.append(post_degree)
        if label == "infrastructure and house utilities":
            infr_post_ids.append(post_id)
            infr_post_sentiments.append(post_sentiment)
            infr_post_sentiment_degrees.append(post_degree)
        if label == "housing costs and property values":
            housing_post_ids.append(post_id)
            housing_post_sentiments.append(post_sentiment)
            housing_post_sentiment_degrees.append(post_degree)
        if label == "economy and jobs":
            econ_post_ids.append(post_id)
            econ_post_sentiments.append(post_sentiment)
            econ_post_sentiment_degrees.append(post_degree)
        if label == "quality of life, noise, and light pollution":
            life_qual_post_ids.append(post_id)
            life_qual_post_sentiments.append(post_sentiment)
            life_qual_post_sentiment_degrees.append(post_degree)
        if label == "environmental impact":
            env_post_ids.append(post_id)
            env_post_sentiments.append(post_sentiment)
            env_post_sentiment_degrees.append(post_degree)
        if label == "government decisions and policies":
            gov_post_ids.append(post_id)
            gov_post_sentiments.append(post_sentiment)
            gov_post_sentiment_degrees.append(post_degree)
        if label == "technology performance and growth":
            tech_post_ids.append(post_id)
            tech_post_sentiments.append(post_sentiment)
            tech_post_sentiment_degrees.append(post_degree)

        a += 1
        print("Posts scanned:", a, end="\r")

print("Total posts scanned:", len(all_post_ids))
print("Total posts with a theme:", matched_posts)
print("Found environmental posts:", len(env_post_ids))
print("Found infrastructure posts:", len(infr_post_ids))
print("Found housing posts:", len(housing_post_ids))
print("Found economic posts:", len(econ_post_ids))
print("Found life quality posts:", len(life_qual_post_ids))
print("Found aesthetic posts:", len(aesth_post_ids))
print("Found government posts:", len(gov_post_ids))
print("Found technological posts:", len(tech_post_ids))
print(not_useful_post_ids)

print(gov_post_sentiments)
print(gov_post_sentiment_degrees)

Total posts scanned: 11
Total posts with a theme: 9
Found environmental posts: 1
Found infrastructure posts: 2
Found housing posts: 0
Found economic posts: 0
Found life quality posts: 0
Found aesthetic posts: 0
Found government posts: 0
Found technological posts: 6
['1889456969771016378', '1881846447353557352']
[]
[]


In [8]:
posts_by_id = {post["id"]: post for post in x_posts}
env_links = []

for id in env_post_ids:
    post = posts_by_id.get(id)
    env_links.append(post["url"])

print(env_links)

['https://x.com/ert_eu/status/1762410996935561249']


In [9]:
theme_lists = [env_post_ids, infr_post_ids, housing_post_ids, econ_post_ids, life_qual_post_ids, aesth_post_ids, gov_post_ids, tech_post_ids]
posts = pd.DataFrame(columns=["ids", "text", "date", "likes", "number of replies", "number of reposts", "views", "environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government", "technology", "AWS", "Amazon", "Google", "Microsoft", "Azure", "Meta", "Oracle", "Equinix", "Digital Realty", "IBM", "Facebook", "Apple", "QTS", "Vantage", "CyrusOne", "CoreSite"])
datacenters_keywords = ["datacenter", "data center", "datacentre", "data centre"]

posts_by_id = {post["id"]: post for post in x_posts}

for theme in theme_lists:
    df1 = pd.DataFrame(columns=["ids", "text", "date", "likes", "number of replies", "number of reposts", "views", "environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government", "technology", "AWS", "Amazon", "Google", "Microsoft", "Azure", "Meta", "Oracle", "Equinix", "Digital Realty", "IBM", "Facebook", "Apple", "QTS", "Vantage", "CyrusOne", "CoreSite"])
    post_ids = []
    post_texts = []
    post_dates = []
    post_likes = []
    post_num_replies = []
    post_num_reposts = []
    post_views = []


    for pid in theme:
        post_ids.append(pid)
        post = posts_by_id.get(pid)
        post_texts.append(remove_links(post["description"]))
        post_dates.append(post["date_posted"])
        post_likes.append(post["likes"])
        post_num_replies.append(post["replies"])
        post_num_reposts.append(post["reposts"])
        if(post["views"] is None):
            post_views.append(0)
        else:
            post_views.append(post["views"])

    df1["ids"] = post_ids
    df1["text"] = post_texts
    df1["date"] = post_dates
    df1["likes"] = post_likes
    df1["number of replies"] = post_num_replies
    df1["number of reposts"] = post_num_reposts
    df1["views"] = post_views

    
    for col in ["environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government", "technology"]:
        df1[col] = False

    for col in ["environment sentiment", "environment sentiment degree", "infrastructure sentiment", "infrastructure sentiment degree", "housing sentiment", "housing sentiment degree", "economy sentiment", "economy sentiment degree", "life quality sentiment", "life quality sentiment degree", "aesthetics sentiment", "aesthetics sentiment degree", "government sentiment", "government sentiment degree", "technology sentiment", "technology sentiment degree"]:
        df1[col] = None

    if theme == env_post_ids:
        df1["environment"] = True
        df1["environment sentiment"] = env_post_sentiments
        df1["environment sentiment degree"] = env_post_sentiment_degrees
    if theme == infr_post_ids:
        df1["infrastructure"] = True
        df1["infrastructure sentiment"] = infr_post_sentiments
        df1["infrastructure sentiment degree"] = infr_post_sentiment_degrees
    if theme == housing_post_ids:
        df1["housing"] = True
        df1["housing sentiment"] = housing_post_sentiments
        df1["housing sentiment degree"] = housing_post_sentiment_degrees
    if theme == econ_post_ids:
        df1["economy"] = True
        df1["economy sentiment"] = econ_post_sentiments
        df1["economy sentiment degree"] = econ_post_sentiment_degrees
    if theme == life_qual_post_ids:
        df1["life quality"] = True
        df1["life quality sentiment"] = life_qual_post_sentiments
        df1["life quality sentiment degree"] = life_qual_post_sentiment_degrees
    if theme == aesth_post_ids:
        df1["aesthetics"] = True
        df1["aesthetics sentiment"] = aesth_post_sentiments
        df1["aesthetics sentiment degree"] = aesth_post_sentiment_degrees
    if theme == gov_post_ids:
        df1["government"] = True
        df1["government sentiment"] = gov_post_sentiments
        df1["government sentiment degree"] = gov_post_sentiment_degrees
    if theme == tech_post_ids:
        df1["technology"] = True
        df1["technology sentiment"] = tech_post_sentiments
        df1["technology sentiment degree"] = tech_post_sentiment_degrees
    posts = pd.concat([posts, df1], ignore_index=True)

/var/folders/9m/h28gbbc970j03ncf7v7dhqk80000gq/T/ipykernel_77607/771045379.py:78: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  posts = pd.concat([posts, df1], ignore_index=True)
/var/folders/9m/h28gbbc970j03ncf7v7dhqk80000gq/T/ipykernel_77607/771045379.py:78: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  posts = pd.concat([posts, df1], ignore_index=True)
/var/folders/9m/h28gbbc970j03ncf7v7dhqk80000gq/T/ipykernel_77607/771045379.py:78: FutureWarning: The behavior of DataFrame concatenation with 

In [11]:
len(posts)

57

In [13]:
posts = posts.astype({
    "ids": "string",
    "text": "string",
    "date": "string",
    "likes": "int64",
    "number of replies": "int64",
    "number of reposts": "int64",
    "views": "int64",
})

In [14]:
grouping_cols = ["ids", "text", "date", "likes", "number of replies", "number of reposts", "views"]

theme_cols = ["environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government", "technology"]
sent_cols  = ["environment sentiment", "environment sentiment degree", "infrastructure sentiment", "infrastructure sentiment degree", "housing sentiment", "housing sentiment degree", "economy sentiment", "economy sentiment degree", "life quality sentiment", "life quality sentiment degree", "aesthetics sentiment", "aesthetics sentiment degree", "government sentiment", "government sentiment degree", "technology sentiment", "technology sentiment degree"]

def first_non_null(s):
    return s.dropna().iloc[0] if s.notna().any() else np.nan

agg = {c: "max" for c in theme_cols}              # True if any True
agg.update({c: first_non_null for c in sent_cols}) # keep the real sentiment if present

posts = posts.groupby(grouping_cols, as_index=False, dropna=False).agg(agg)

In [15]:
len(posts)

52

In [ ]:
posts.to_json('x_ABSA_entire_dataframe.json', orient='records', indent=4)

In [16]:
posts.head(30)

,ids,text,date,likes,number of replies,number of reposts,views,environment,infrastructure,housing,...,economy sentiment,economy sentiment degree,life quality sentiment,life quality sentiment degree,aesthetics sentiment,aesthetics sentiment degree,government sentiment,government sentiment degree,technology sentiment,technology sentiment degree
0,1517518685761638401,"At PayPal, we’ve matched 100% of the energy in...",2022-04-22T15:00:23.000Z,53,34,10,0,True,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1557124752002670599,"Safe Host, a data centre in Switzerland, uses ...",2022-08-09T22:00:45.000Z,34,1,2,0,False,False,False,...,NaN,NaN,NaN,NaN,Neutral,0.570602,NaN,NaN,NaN,NaN
2,1658823223906037760,How Power Quality Intelligence Can Drive Data ...,2023-05-17T13:14:10.000Z,0,0,1,0,False,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Positive,0.723553
3,1762410996935561249,"ERT Member @Cheydema, CEO @orange, talks about...",2024-02-27T09:35:00.000Z,0,0,0,117,True,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1800890805701239202,"AMD, along with other industry leaders, have a...",2024-06-12T14:00:01.000Z,95,1,9,18455,False,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Positive,0.896538
5,1814308326731067544,We're thrilled to welcome Andras Szakonyi as o...,2024-07-19T14:36:27.000Z,9,1,3,307,False,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Positive,0.986817
6,1819284026135363757,🌿Reduced Costs\n🌿Improved Performance\n🌿Lower ...,2024-08-02T08:08:07.000Z,23,0,11,4126,False,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Positive,0.986949
7,1823088090589638823,Join us for a conversation discussing the futu...,2024-08-12T20:04:06.000Z,14,0,2,994,False,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Positive,0.893870
8,1824265950822031421,2 bullish concalls on Wednesday:\n\nConcall hi...,2024-08-16T02:04:30.000Z,510,19,59,72175,False,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Positive,0.785991
9,1825674604267892916,Demand for AI is driving data center water con...,2024-08-19T23:21:59.000Z,33,5,15,30801,False,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Negative,0.984759


In [17]:
print(posts.columns)

Index(['ids', 'text', 'date', 'likes', 'number of replies',
       'number of reposts', 'views', 'environment', 'infrastructure',
       'housing', 'economy', 'life quality', 'aesthetics', 'government',
       'technology', 'environment sentiment', 'environment sentiment degree',
       'infrastructure sentiment', 'infrastructure sentiment degree',
       'housing sentiment', 'housing sentiment degree', 'economy sentiment',
       'economy sentiment degree', 'life quality sentiment',
       'life quality sentiment degree', 'aesthetics sentiment',
       'aesthetics sentiment degree', 'government sentiment',
       'government sentiment degree', 'technology sentiment',
       'technology sentiment degree'],
      dtype='object')


In [18]:
# calculating average sentiment based on theme:
# Weigh all posts by their degree in the numerator and denominator, means that the average sentiment will just be +/- 1 if there are only positive or negative themes but other than that does a pretty good job of weighing neutrality
def avg_sentiment_calculation(theme):
    theme_posts = posts[posts[theme] == True]
    if len(theme_posts) > 0:
        pos = theme_posts.loc[theme_posts[f"{theme} sentiment"] == "Positive", f"{theme} sentiment degree"].sum()
        neg = theme_posts.loc[theme_posts[f"{theme} sentiment"] == "Negative", f"{theme} sentiment degree"].sum()
        total = theme_posts[f"{theme} sentiment degree"].sum()
        return len(theme_posts), (pos-neg)/total
    else:
        return 0, None


print("Number of environmental posts: ", avg_sentiment_calculation("environment")[0], ", Average sentiment of environmental posts: ", avg_sentiment_calculation("environment")[1], sep="")
print("Number of infrastructural posts: ", avg_sentiment_calculation("infrastructure")[0], ", Average sentiment of infrastructural posts: ", avg_sentiment_calculation("infrastructure")[1], sep="")
print("Number of housing-related posts: ", avg_sentiment_calculation("housing")[0], ", Average sentiment of housing-related posts: ", avg_sentiment_calculation("housing")[1], sep="")
print("Number of economic posts: ", avg_sentiment_calculation("economy")[0], ", Average sentiment of economic posts: ", avg_sentiment_calculation("economy")[1], sep="")
print("Number of life-quality-related posts: ", avg_sentiment_calculation("life quality")[0], ", Average sentiment of life-quality-related posts: ", avg_sentiment_calculation("life quality")[1], sep="")
print("Number of aesthetics-related posts: ", avg_sentiment_calculation("aesthetics")[0], ", Average sentiment of aesthetics-related posts: ", avg_sentiment_calculation("aesthetics")[1], sep="")
print("Number of governmental posts: ", avg_sentiment_calculation("government")[0], ", Average sentiment of governmental posts: ", avg_sentiment_calculation("government")[1], sep="")
print("Number of technological posts: ", avg_sentiment_calculation("technology")[0], ", Average sentiment of technological posts: ", avg_sentiment_calculation("technology")[1], sep="")


Number of environmental posts: 4, Average sentiment of environmental posts: 0.0192378281800267
Number of infrastructural posts: 3, Average sentiment of infrastructural posts: 0.0
Number of housing-related posts: 0, Average sentiment of housing-related posts: None
Number of economic posts: 1, Average sentiment of economic posts: 1.0
Number of life-quality-related posts: 0, Average sentiment of life-quality-related posts: None
Number of aesthetics-related posts: 2, Average sentiment of aesthetics-related posts: 0.5969432276903854
Number of governmental posts: 2, Average sentiment of governmental posts: 0.48362021412520195
Number of technological posts: 45, Average sentiment of technological posts: 0.7291329816359197


In [19]:
posts['year'] = pd.to_datetime(posts['date']).dt.year

year_datasets = {year: posts[posts['year'] == year] for year in range(2010, 2027)}

posts_2010 = year_datasets[2010]
posts_2020 = year_datasets[2020]

In [33]:
env_posts = posts.loc[posts["environment"]]
env_posts = env_posts.drop(columns=["ids", "date", "likes", "number of replies", "number of reposts", "views", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government", "technology"])
env_posts.head()

,text,environment,environment sentiment,environment sentiment degree,infrastructure sentiment,infrastructure sentiment degree,housing sentiment,housing sentiment degree,economy sentiment,economy sentiment degree,life quality sentiment,life quality sentiment degree,aesthetics sentiment,aesthetics sentiment degree,government sentiment,government sentiment degree,technology sentiment,technology sentiment degree
0,"At PayPal, we’ve matched 100% of the energy in...",True,Neutral,0.636376,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"ERT Member @Cheydema, CEO @orange, talks about...",True,Positive,0.861934,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25,Trump targets key environmental law in bid to ...,True,Negative,0.804779,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Neutral,0.845943,Positive,0.874794
39,"""We are really looking at our water use with d...",True,Neutral,0.667844,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
